# Survey of Consumer Finances (SCF)  
This notebook is going to be the first notebook in exploring the summarized version of the Survey of Consumer Finances. The summarized version is found on the website: https://www.federalreserve.gov/econres/scfindex.htm . We have decided to explore the summarized version first, as it has been cleaned and summarized by the Fed. They have actually created five different sets of the dataset with differing imputed values for missing values. We will need to pick one of the versions or potentially average the results from all five. Let's start by importing our libraries and downloading the dataset if don't yet have it.

## Libraries


In [ ]:
import requests
import zipfile
import io

import numpy as np
import pandas as pd
import altair as alt

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

url = "https://www.federalreserve.gov/econres/files/scfp2022excel.zip"

## Data Download or Import
Alright, now that we have all of this, I want to see if we have the file. If we have the file, we can just import it, however if we haven't downloaded it yet, we will go to the website and download it. 

In [ ]:
# Check to see if the data file already exists before attempting to download
try:
    print("Importing data from the Survey of Consumer Finances...")
    data = pd.read_csv("data/SCFP2022.csv")
    print("Data imported successfully.")

#If it doesn't exist we can download it form the Federal Reserve's website
except:
    print("File not found. Downloading survey data from the Federal Reserve's website...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    print(f"Status code: {response.status_code}") # Check if the request was successful
    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall("data")  # Extracts all files in the zip to the current directory
        print("Data downloaded and extracted successfully.")
        data = pd.read_csv("data/SCFP2022.csv")
    else:
        print("Failed to download data. Please check the URL and try again.")


Importing data from the Survey of Consumer Finances...
Data imported successfully.


## Inital Exploration
To start, we should just get a sense of the amount of data that we have in our dataframe. What missing values we have, the columns, we have, descriptive statistics etc. 

In [30]:
print(f"Dataframe shape: {data.shape}")
data.head(10)

Dataframe shape: (22975, 358)


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT,equity_alloc
0,1,11,3027.956120,2,70,5,9,3,2,2,...,2,4,2,1,8,3,3,2,1,0.051196
1,1,12,3054.900065,2,70,5,9,3,2,2,...,2,5,2,1,8,3,3,2,1,0.165839
2,1,13,3163.637766,2,70,5,9,3,2,2,...,2,4,2,1,8,3,3,1,1,0.044400
3,1,14,3166.228463,2,70,5,9,3,2,2,...,2,4,1,1,6,3,2,1,1,0.050688
4,1,15,3235.624715,2,70,5,9,3,2,2,...,2,4,2,1,8,3,3,1,1,0.032703
5,2,21,236.634754,1,46,3,12,4,2,0,...,5,5,5,2,8,9,9,4,4,0.024184
6,2,22,245.848398,1,46,3,12,4,2,0,...,5,5,5,2,8,9,9,4,4,0.012072
7,2,23,253.103477,1,46,3,12,4,2,0,...,5,5,5,2,8,9,9,4,4,0.052373
8,2,24,252.908118,1,46,3,12,4,2,0,...,5,5,5,2,8,9,9,4,4,0.060289
9,2,25,253.811312,1,46,3,12,4,2,0,...,5,5,5,2,8,9,9,4,4,0.004028


In [15]:
columns = data.columns.tolist()
print(columns)

['YY1', 'Y1', 'WGT', 'HHSEX', 'AGE', 'AGECL', 'EDUC', 'EDCL', 'MARRIED', 'KIDS', 'LF', 'LIFECL', 'FAMSTRUCT', 'RACECL', 'RACECL4', 'RACECL5', 'RACECL_EX', 'RACE', 'OCCAT1', 'OCCAT2', 'INDCAT', 'FOODHOME', 'FOODAWAY', 'FOODDELV', 'RENT', 'INCOME', 'WAGEINC', 'BUSSEFARMINC', 'INTDIVINC', 'KGINC', 'SSRETINC', 'TRANSFOTHINC', 'PENACCTWD', 'NORMINC', 'WSAVED', 'SAVED', 'SAVRES1', 'SAVRES2', 'SAVRES3', 'SAVRES4', 'SAVRES5', 'SAVRES6', 'SAVRES7', 'SAVRES8', 'SAVRES9', 'SPENDMOR', 'SPENDLESS', 'EXPENSHILO', 'LATE', 'LATE60', 'HPAYDAY', 'BNKRUPLAST5', 'KNOWL', 'YESFINRISK', 'NOFINRISK', 'CRDAPP', 'TURNDOWN', 'FEARDENIAL', 'TURNFEAR', 'FORECLLAST5', 'EMERGBORR', 'EMERGSAV', 'EMERGPSTP', 'EMERGCUT', 'EMERGWORK', 'HBORRFF', 'HBORRCC', 'HBORRALT', 'HBORRFIN', 'HSAVFIN', 'HSAVNFIN', 'HPSTPPAY', 'HPSTPLN', 'HPSTPOTH', 'HCUTFOOD', 'HCUTENT', 'HCUTOTH', 'FINLIT', 'BSHOPNONE', 'BSHOPGRDL', 'BSHOPMODR', 'ISHOPNONE', 'ISHOPGRDL', 'ISHOPMODR', 'BCALL', 'BMAGZNEWS', 'BMAILADTV', 'BINTERNET', 'BFRIENDWORK', 

Alright, 357 columns is going to be way more than we need. So let's take a look at the documentation on the website to see which ones will likely be useful and which likely won't be. Some things I learned when reading:

YESFINRISK/NOFINRISK - Self reported risk tolerance derived from survey  
STOCKS — direct stock ownership  
STMUTF — stock mutual funds  
EQUITY — total equity holdings  
RETQLIQ — retirement account value  

Further, we can construct our own risk tolerance metric using some of these other:  
EQUITY, DEQ, RETEQ — equity holdings (total, direct, retirement)
STOCKS, HSTOCKS — stock ownership (value and indicator)
LEVRATIO — leverage ratio (debt/assets) — interesting risk signal
DEBT2INC — debt-to-income ratio

Portfolio composition signals:

STMUTF, TFBMUTF, GBMUTF, OBMUTF — mutual fund types
BOND, RETQLIQ — fixed income and retirement liquidity  


Maybe look for things like life insurance to show lower risk proxies etc. 

In [23]:
print(data['YESFINRISK'].value_counts())
print(data['NOFINRISK'].value_counts())

YESFINRISK
0    21754
1     1221
Name: count, dtype: int64
NOFINRISK
0    15679
1     7296
Name: count, dtype: int64


Alright, financial risk would likely be highly imbalanced (~5% of households are explicitly willing to take financial risk and ~32% are unwilling). The rest are likely our moderate risk group but this is still too imbalanced. We can potentially calculate financial risk tolerance by looking how much of families assets are invested in equities. Let's take a look at what this would look like. 

In [31]:
print(data['YY1'].value_counts())
print(data['Y1'].value_counts())

YY1
1       5
2       5
3       5
4       5
5       5
       ..
4599    5
4600    5
4601    5
4602    5
4603    5
Name: count, Length: 4595, dtype: int64
Y1
11       1
12       1
13       1
14       1
15       1
        ..
46031    1
46032    1
46033    1
46034    1
46035    1
Name: count, Length: 22975, dtype: int64


Alright, it is important to note that YY1 is the family identifier and that Y1 is a concatinated family identifier and the imputation. We will need to filter it so that we only take one of the five imputation forms. 

In [32]:
df = data.copy()
df['imputation'] = df['Y1'].astype(str).str[-1]
df = df[df['imputation'] == '1']
print(f"Dataframe Shape after filtering for first imputation: {df.shape}")
df['equity_alloc'] = df['EQUITY'] / df['ASSET']
df['equity_alloc'] = df['equity_alloc'].clip(0, 1)
df['equity_alloc'].describe()

Dataframe Shape after filtering for first imputation: (4595, 359)


count    4574.000000
mean        0.142481
std         0.203370
min         0.000000
25%         0.000000
50%         0.038033
75%         0.226328
max         0.995591
Name: equity_alloc, dtype: float64

This is looking right and very interesting. We were expecting ~4600 families. We have 4574. An average equity investment is 14% of total assets with a low being 0 and high being 99.5%. I think this may be a very good way of assessing risk. I'm going to take a bit of a break and come back to this.